In [ ]:
%pip install gwpy gwosc scipy # to get noise data
%pip install pycbc # to generate waveforms

the above lines install the necessary packages to access the noise time series data and the gw signal generator function.

the code below imports the necessary functions/libraries and finds the start and end times of the 03a run and 03b runs. it uses these times to retreive and print all the real gw events recorded during those runs. as shown in the output below, many events were recorded, so clean noise segments can not be determined manually.

In [2]:
from gwpy.timeseries import TimeSeries
from gwosc.datasets import event_gps
from gwosc.datasets import find_datasets, run_segment

# get 03a, set 03a boundaries
start_o3a, end_o3a = run_segment("O3a")
print(start_o3a, end_o3a)
print()
# all confirmed 03a event names
events_o3a = find_datasets(type="event", segment=(start_o3a, end_o3a))
# event names + event times
event_times_03a = [event_gps(name) for name in events_o3a]
print(event_times_03a)
print()
print()
# get 03b, set 03b boundaries
start_o3b, end_o3b = run_segment("O3b")
print(start_o3b, end_o3b)
print()
# all confirmed 03b event names
events_o3b = find_datasets(type="event", segment=(start_o3b, end_o3b))
# event names + event times
event_times_03b = [event_gps(name) for name in events_o3b]
print(event_times_03b)

/Users/roxydhyana/miniconda3/lib/python3.13/site-packages/gwpy/time/_ligotimegps.py:42: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  from lal import LIGOTimeGPS


1238166018 1253977218

[1253402833.0, 1238303737.2, 1238782700.3, 1238782700.2, 1238782700.3, 1239082262.2, 1239082262.2, 1239082262.2, 1239082262.1, 1239082262.1, 1239168612.5, 1239168612.5, 1239168612.5, 1239198206.7, 1239198206.7, 1239198206.7, 1239917954.3, 1239917954.2, 1239917954.2, 1240164426.1, 1240164426.1, 1240215503.0, 1240215503.0, 1240215503.0, 1240327333.3, 1240327333.3, 1240340820.6, 1240944862.3, 1240944862.2, 1240944862.3, 1241719652.4, 1241719652.4, 1241719652.4, 1241816086.8, 1241816086.7, 1241816086.7, 1241852074.8, 1241852074.8, 1241852074.8, 1242107479.8, 1242107479.8, 1242107479.8, 1242315362.4, 1242315362.3, 1242315362.4, 1242442967.4, 1242442967.4, 1242442967.4, 1242442967.4, 1242459857.5, 1242459857.4, 1242459857.4, 1242984073.8, 1242984073.7, 1242984073.8, 1243305426.3, 1243533585.1, 1243533585.0, 1243533585.1, 1245035079.3, 1245035079.3, 1245955943.2, 1245955943.1, 1246048404.6, 1246048404.5, 1246272532.7, 1246487219.3, 1246487219.3, 1246487219.3, 1246523564

the two functions below analytically determine clean noise segments consistent across all 3 detectors.

the code below defines the function find_clean_segments which identifies n candidate segments within a given observing run that do not overlap with any confirmed GW event. the function all_detectors_online checks each candidate segment against all 3 detectors, keeping only the segments where H1, L1, and V1 were all simultaneously recording data.

In [10]:
from gwosc.timeline import get_segments

def find_clean_segments(start, end, duration, event_times, buffer=60, n_segments=12):
    segments = []
    t = start
    spacing = (end-start) // (n_segments)
    while len(segments) < n_segments and t + duration < end:
        segment_end = t + duration
        buffer_start = t - buffer
        buffer_end = segment_end + buffer
        if all(not (buffer_start < event < buffer_end) for event in event_times):
            segments.append((t, segment_end))
            t += spacing  # move past this window
        else:
            t += 60  # skip ahead and try again
    return segments

def all_detectors_online(start, end):
    for ifo in ["H1", "L1", "V1"]:
        segs = get_segments(f"{ifo}_DATA", start, end)
        if not segs or segs[0] != (start, end):
            return False
    return True

clean_noise_o3a = find_clean_segments(start_o3a, end_o3a, 300, event_times_03a)

noise_data_o3a = {"H1": [], "L1": [], "V1": []}
noise_array_o3a = {"H1": [], "L1": [], "V1": []} # gwpy TimeSeries -> numpy array

for start, end in clean_noise_o3a:  
    if all_detectors_online(start, end):
        for ifo in ["H1", "L1", "V1"]:
            data = TimeSeries.fetch_open_data(ifo, start, end, cache=True)
            noise_data_o3a[ifo].append(data)
            noise_array_o3a[ifo].append(data.value)
        print(f"Using {start}-{end}")
    else:
        print(f"Skipping {start}-{end}, not all detectors online")

total_segments_o3a = len(noise_data_o3a["H1"])
total_seconds_o3a = total_segments * 300
print(f"Total seconds: {total_seconds_o3a}")

Using 1238166018-1238166318
Skipping 1239483618-1239483918, not all detectors online
Using 1240801218-1240801518
Skipping 1242118818-1242119118, not all detectors online
Using 1243436418-1243436718
Using 1244754018-1244754318
Skipping 1246071618-1246071918, not all detectors online
Using 1247389218-1247389518
Using 1248706818-1248707118
Skipping 1250024418-1250024718, not all detectors online
Using 1251342018-1251342318
Using 1252659618-1252659918
Total seconds: 2400


In [12]:
clean_noise_o3b = find_clean_segments(start_o3b, end_o3b, 300, event_times_03b, n_segments=14)

noise_data_o3b = {"H1": [], "L1": [], "V1": []}
noise_array_o3b = {"H1": [], "L1": [], "V1": []} # gwpy TimeSeries -> numpy array

for start, end in clean_noise_o3b:  
    if all_detectors_online(start, end):
        for ifo in ["H1", "L1", "V1"]:
            data = TimeSeries.fetch_open_data(ifo, start, end, cache=True)
            noise_data_o3b[ifo].append(data)
            noise_array_o3b[ifo].append(data.value)
        print(f"Using {start}-{end}")
    else:
        print(f"Skipping {start}-{end}, not all detectors online")

total_segments_o3b = len(noise_data_o3b["H1"])
total_seconds_o3b = total_segments_o3b * 300
print(f"Total seconds: {total_seconds_o3b}")

Skipping 1256655618-1256655918, not all detectors online
Using 1257563332-1257563632
Using 1258471046-1258471346
Using 1259378760-1259379060
Skipping 1260286474-1260286774, not all detectors online
Using 1261194188-1261194488
Using 1262101902-1262102202
Using 1263009616-1263009916
Using 1263917330-1263917630
Skipping 1264825044-1264825344, not all detectors online
Using 1265732758-1265733058
Using 1266640472-1266640772
Skipping 1267548186-1267548486, not all detectors online
Using 1268455900-1268456200
Total seconds: 3000
